# Analisis Penjualan dan Risiko Keterlambatan Pembayaran

## 1. Latar Belakang dan Rumusan Masalah

### Latar Belakang
Sebuah distributor (simulasi) mencatat transaksi penjualan sepanjang 12 bulan. Manajemen melihat bahwa total penjualan meningkat dari bulan ke bulan, namun laba bersih tidak selalu ikut naik. Selain itu, beberapa pelanggan terlambat membayar sehingga arus kas terganggu. Diperlukan analisis untuk memahami penyebab dan mengidentifikasi pelanggan berisiko.

### Rumusan Masalah (CRISP-DM — Pemahaman Bisnis)
1. **Mengapa kenaikan penjualan tidak selalu diikuti kenaikan laba?** — Membutuhkan analitik deskriptif dan diagnostik untuk melihat hubungan penjualan, harga pokok, dan biaya operasional.
2. **Pelanggan mana yang berisiko terlambat membayar?** — Membutuhkan analitik prediktif menggunakan informasi yang tersedia sebelum pembayaran.

## 2. Tujuan dan Batasan Studi Kasus

### Tujuan
1. Menganalisis tren penjualan dan laba bulanan untuk menemukan bulan dengan penjualan tinggi tetapi laba rendah.
2. Membangun model klasifikasi sederhana untuk memprediksi risiko keterlambatan pembayaran.
3. Menunjukkan alur kerja analisis data sesuai CRISP-DM secara sederhana.

### Batasan
- Semua data bersifat **sintetis** (simulasi), bukan data perusahaan asli.
- Model menggunakan **LogisticRegression** sederhana melalui Pipeline, dibandingkan dengan **DummyClassifier** sebagai baseline.
- Fitur prediksi hanya menggunakan informasi yang tersedia **sebelum** pembayaran terjadi.
- Tidak ada deployment, API, database server, atau fitur kompleks lainnya.
- Hasil bergantung pada seed dan data sintetis; tidak menjamin performa pada data nyata.

## 4. Dataset, Fitur, dan Asumsi

### Deskripsi Dataset
Dataset terdiri dari **300 transaksi sintetis** selama 12 bulan yang dibangkitkan menggunakan `seed=42`.

### Kolom/Fitur

| Kolom | Deskripsi | Tipe |
|---|---|---|
| `tanggal` | Tanggal transaksi (Januari-Desember) | datetime |
| `id_pelanggan` | ID unik pelanggan (P001-P030) | kategorikal |
| `nilai_penjualan` | Nilai penjualan dalam rupiah (simulasi) | numerik |
| `harga_pokok` | Harga pokok penjualan (~60-85% dari nilai penjualan) | numerik |
| `biaya_operasional` | Biaya operasional teralokasi per transaksi | numerik |
| `termin_kredit` | Jangka waktu kredit dalam hari (14, 30, 45, 60) | numerik |
| `jumlah_terlambat_sebelumnya` | Berapa kali pelanggan ini terlambat bayar sebelumnya | numerik |
| `terlambat` | Label: 1 = terlambat bayar, 0 = tepat waktu | biner |

### Rumus Laba
```
laba = nilai_penjualan - harga_pokok - biaya_operasional
```

### Asumsi
- Data sintetis dibuat untuk keperluan pembelajaran, **bukan** data survei atau data perusahaan asli.
- Distribusi harga pokok dan biaya operasional bervariasi sehingga ada transaksi berlaba negatif.
- Proporsi pelanggan terlambat sekitar 25-30% untuk mensimulasikan ketidakseimbangan ringan.
- Beberapa nilai kosong dan duplikasi ditambahkan secara sengaja untuk latihan pembersihan data.

## 4. Metode dan Alur Penyelesaian

Alur kerja mengikuti kerangka **CRISP-DM** yang disederhanakan:

1. **Pemahaman Bisnis** — Rumusan masalah dan tujuan analisis.
2. **Pemahaman Data** — Bangkitkan data sintetis, periksa dimensi, tipe, dan distribusi.
3. **Persiapan Data** — Tambahkan ketidaksempurnaan (null, duplikat), lalu bersihkan. Hitung laba.
4. **Analisis Deskriptif-Diagnostik** — Ringkasan per bulan, grafik tren, insight.
5. **Pemodelan Prediktif** — Pipeline (StandardScaler + LogisticRegression) vs DummyClassifier, split stratified 70:30.
6. **Evaluasi** — Confusion matrix, precision, recall, F1, akurasi.
7. **Pembahasan & Kesimpulan** — Jawaban atas rumusan masalah, keterbatasan.

## 5. Implementasi Python

### 6.1 Import Library

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, precision_score, recall_score, f1_score,
    ConfusionMatrixDisplay
)
import warnings
warnings.filterwarnings('ignore')

print("Semua library berhasil diimpor.")

### 6.2 Bangkitkan Data Sintetis (300 Transaksi, 12 Bulan)

Data ini sepenuhnya **simulasi** menggunakan `seed=42`. Bukan data lapangan atau data perusahaan asli.

In [ ]:
np.random.seed(42)

n = 300
pelanggan_list = [f"P{str(i).zfill(3)}" for i in range(1, 31)]  # 30 pelanggan

# Tanggal acak selama 12 bulan (2025)
start_date = pd.Timestamp("2025-01-01")
end_date = pd.Timestamp("2025-12-31")
tanggal = pd.to_datetime(
    np.random.randint(start_date.value // 10**9, end_date.value // 10**9, size=n),
    unit='s'
)

id_pelanggan = np.random.choice(pelanggan_list, size=n)
nilai_penjualan = np.random.uniform(500_000, 15_000_000, size=n).round(-3)
rasio_hpp = np.random.uniform(0.60, 0.85, size=n)
harga_pokok = (nilai_penjualan * rasio_hpp).round(-3)
biaya_operasional = np.random.uniform(50_000, 2_000_000, size=n).round(-3)
termin_kredit = np.random.choice([14, 30, 45, 60], size=n, p=[0.2, 0.4, 0.25, 0.15])

jumlah_terlambat_sblm = np.random.choice(
    [0, 1, 2, 3, 4, 5], size=n,
    p=[0.40, 0.25, 0.15, 0.10, 0.06, 0.04]
)

# Label terlambat: dipengaruhi termin panjang dan riwayat keterlambatan
prob_terlambat = 0.05 + 0.08 * jumlah_terlambat_sblm + 0.003 * termin_kredit
prob_terlambat = np.clip(prob_terlambat, 0, 0.95)
terlambat = np.random.binomial(1, prob_terlambat)

df = pd.DataFrame({
    'tanggal': tanggal,
    'id_pelanggan': id_pelanggan,
    'nilai_penjualan': nilai_penjualan,
    'harga_pokok': harga_pokok,
    'biaya_operasional': biaya_operasional,
    'termin_kredit': termin_kredit,
    'jumlah_terlambat_sebelumnya': jumlah_terlambat_sblm,
    'terlambat': terlambat
})

df['laba'] = df['nilai_penjualan'] - df['harga_pokok'] - df['biaya_operasional']

print(f"Dataset awal: {df.shape[0]} baris, {df.shape[1]} kolom")
print(f"Proporsi terlambat: {df['terlambat'].mean():.2%}")
df.head(10)

### 6.3 Tambahkan Ketidaksempurnaan Data (Null dan Duplikasi)

Sesuai ketentuan, kita menambahkan beberapa nilai kosong dan baris duplikat secara sengaja untuk mendemonstrasikan pembersihan data.

In [ ]:
np.random.seed(42)

# Tambah 10 nilai kosong pada kolom nilai_penjualan
idx_null_penjualan = np.random.choice(df.index, size=10, replace=False)
df.loc[idx_null_penjualan, 'nilai_penjualan'] = np.nan

# Tambah 8 nilai kosong pada kolom biaya_operasional
idx_null_biaya = np.random.choice(df.index, size=8, replace=False)
df.loc[idx_null_biaya, 'biaya_operasional'] = np.nan

# Tambah 15 baris duplikat
idx_duplikat = np.random.choice(df.index, size=15, replace=False)
df_duplikat = df.loc[idx_duplikat].copy()
df = pd.concat([df, df_duplikat], ignore_index=True)

print(f"Dataset setelah penambahan noise:")
print(f"  Jumlah baris: {df.shape[0]}")
print(f"  Nilai kosong per kolom:")
print(df.isnull().sum().to_string())
print(f"\n  Jumlah baris duplikat: {df.duplicated().sum()}")

### 6.4 Pembersihan Data (CRISP-DM: Persiapan Data)

Langkah pembersihan:
1. Hapus baris duplikat.
2. Hapus baris dengan nilai kosong (karena jumlahnya kecil dibanding total data).
3. Hitung ulang kolom `laba`.

In [ ]:
# Hapus duplikat
df = df.drop_duplicates().reset_index(drop=True)
print(f"Setelah hapus duplikat: {df.shape[0]} baris")

# Hapus baris dengan nilai kosong
df = df.dropna().reset_index(drop=True)
print(f"Setelah hapus null: {df.shape[0]} baris")

# Hitung ulang laba
df['laba'] = df['nilai_penjualan'] - df['harga_pokok'] - df['biaya_operasional']

print(f"\nRingkasan setelah pembersihan:")
print(f"  Nilai kosong: {df.isnull().sum().sum()}")
print(f"  Duplikat: {df.duplicated().sum()}")
print(f"  Jumlah baris akhir: {df.shape[0]}")
df.describe().round(0)

### 6.5 Analisis Deskriptif — Ringkasan Per Bulan

In [ ]:
df['bulan'] = df['tanggal'].dt.month

ringkasan_bulanan = df.groupby('bulan').agg(
    jumlah_transaksi=('nilai_penjualan', 'count'),
    total_penjualan=('nilai_penjualan', 'sum'),
    total_harga_pokok=('harga_pokok', 'sum'),
    total_biaya_ops=('biaya_operasional', 'sum'),
    total_laba=('laba', 'sum'),
    rata_rata_laba=('laba', 'mean'),
    persen_terlambat=('terlambat', 'mean')
).reset_index()

ringkasan_bulanan['persen_terlambat'] = (ringkasan_bulanan['persen_terlambat'] * 100).round(1)

nama_bulan = {1:'Jan', 2:'Feb', 3:'Mar', 4:'Apr', 5:'Mei', 6:'Jun',
              7:'Jul', 8:'Agt', 9:'Sep', 10:'Okt', 11:'Nov', 12:'Des'}
ringkasan_bulanan['nama_bulan'] = ringkasan_bulanan['bulan'].map(nama_bulan)

print("Ringkasan Penjualan dan Laba per Bulan:")
ringkasan_bulanan[['nama_bulan', 'jumlah_transaksi', 'total_penjualan',
                    'total_laba', 'rata_rata_laba', 'persen_terlambat']]

### 6.6 Grafik 1 — Tren Penjualan dan Laba Bulanan

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 5))
x = ringkasan_bulanan['nama_bulan']

ax1.bar(x, ringkasan_bulanan['total_penjualan'] / 1e6,
        color='steelblue', alpha=0.7, label='Total Penjualan (Juta Rp)')
ax1.set_xlabel('Bulan')
ax1.set_ylabel('Total Penjualan (Juta Rp)', color='steelblue')
ax1.tick_params(axis='y', labelcolor='steelblue')

ax2 = ax1.twinx()
ax2.plot(x, ringkasan_bulanan['total_laba'] / 1e6,
         color='orangered', marker='o', linewidth=2, label='Total Laba (Juta Rp)')
ax2.set_ylabel('Total Laba (Juta Rp)', color='orangered')
ax2.tick_params(axis='y', labelcolor='orangered')
ax2.axhline(y=0, color='gray', linestyle='--', linewidth=0.8)

plt.title('Tren Penjualan vs Laba Bulanan')
fig.legend(loc='upper left', bbox_to_anchor=(0.12, 0.95))
fig.tight_layout()
plt.savefig('grafik_penjualan_laba.png', dpi=150, bbox_inches='tight')
print("Grafik disimpan sebagai 'grafik_penjualan_laba.png'")
plt.show()

### 6.7 Grafik 2 — Distribusi Keterlambatan Pembayaran per Termin Kredit

In [ ]:
terlambat_termin = df.groupby('termin_kredit')['terlambat'].agg(['sum', 'count']).reset_index()
terlambat_termin.columns = ['termin_kredit', 'jumlah_terlambat', 'total']
terlambat_termin['persen_terlambat'] = (terlambat_termin['jumlah_terlambat'] /
                                         terlambat_termin['total'] * 100).round(1)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(terlambat_termin['termin_kredit'].astype(str),
              terlambat_termin['persen_terlambat'],
              color=['#2ecc71', '#f39c12', '#e67e22', '#e74c3c'], edgecolor='white')

for bar, val in zip(bars, terlambat_termin['persen_terlambat']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val}%', ha='center', va='bottom', fontweight='bold')

ax.set_xlabel('Termin Kredit (Hari)')
ax.set_ylabel('Persentase Terlambat (%)')
ax.set_title('Persentase Keterlambatan Pembayaran per Termin Kredit')
ax.set_ylim(0, max(terlambat_termin['persen_terlambat']) + 10)
plt.tight_layout()
plt.savefig('grafik_terlambat_termin.png', dpi=150, bbox_inches='tight')
print("Grafik disimpan sebagai 'grafik_terlambat_termin.png'")
plt.show()

### 6.8 Insight dari Analisis Deskriptif

> **Catatan:** Insight berikut harus diverifikasi setelah menjalankan kode di Google Colab. Angka-angka spesifik akan muncul dari output kode, bukan dikarang.

**Tiga insight yang diharapkan muncul dari data:**

1. **Penjualan tinggi tidak selalu menghasilkan laba tinggi.** Beberapa bulan menunjukkan total penjualan besar tetapi laba relatif rendah atau negatif. Ini disebabkan variasi harga pokok (60-85%) dan biaya operasional yang tidak proporsional. Sesuai contoh pemateri bahwa peningkatan penjualan tidak otomatis berarti kinerja membaik.

2. **Termin kredit panjang cenderung memiliki persentase keterlambatan lebih tinggi.** Pelanggan dengan termin 45-60 hari menunjukkan proporsi terlambat lebih besar dibanding termin 14 hari.

3. **Riwayat keterlambatan sebelumnya menjadi indikator kuat.** Pelanggan dengan `jumlah_terlambat_sebelumnya` tinggi cenderung lebih sering terlambat. Ini mendukung penggunaan fitur historis dalam model prediktif.

> **Penting:** Verifikasi insight ini dengan output grafik dan tabel yang dihasilkan saat menjalankan notebook.

### 6.9 Pemodelan — Persiapan Fitur dan Pembagian Data

**Fitur yang digunakan** (tersedia sebelum pembayaran):
- `nilai_penjualan` — besaran tagihan
- `termin_kredit` — jangka waktu pembayaran
- `jumlah_terlambat_sebelumnya` — riwayat keterlambatan pelanggan

**Fitur yang TIDAK digunakan** (mencegah kebocoran data):
- Tanggal pelunasan, jumlah hari terlambat, status pembayaran aktual — baru diketahui *setelah* pembayaran terjadi.

In [ ]:
fitur_cols = ['nilai_penjualan', 'termin_kredit', 'jumlah_terlambat_sebelumnya']
X = df[fitur_cols].copy()
y = df['terlambat'].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"Data latih: {X_train.shape[0]} baris")
print(f"Data uji  : {X_test.shape[0]} baris")
print(f"\nDistribusi kelas pada data latih:")
print(y_train.value_counts().to_string())
print(f"\nDistribusi kelas pada data uji:")
print(y_test.value_counts().to_string())

### 6.10 Pelatihan Model: Pipeline LogisticRegression vs DummyClassifier

In [ ]:
# Pipeline: StandardScaler + LogisticRegression
pipe_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(random_state=42, max_iter=1000))
])

# Baseline: DummyClassifier (strategi most_frequent)
pipe_dummy = DummyClassifier(strategy='most_frequent', random_state=42)

pipe_lr.fit(X_train, y_train)
pipe_dummy.fit(X_train, y_train)

print("Model berhasil dilatih.")
print("  LogisticRegression Pipeline: selesai")
print("  DummyClassifier (most_frequent): selesai")

### 6.11 Evaluasi Model

In [ ]:
def evaluasi_model(nama, model, X_test, y_test):
    """Hitung dan tampilkan metrik evaluasi."""
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    print(f"\n{'='*50}")
    print(f"Model: {nama}")
    print(f"{'='*50}")
    print(f"Akurasi  : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1-Score : {f1:.4f}")
    print(f"\nClassification Report:")
    print(classification_report(y_test, y_pred,
          target_names=['Tepat Waktu', 'Terlambat'], zero_division=0))
    return y_pred

y_pred_lr = evaluasi_model("LogisticRegression (Pipeline)", pipe_lr, X_test, y_test)
y_pred_dummy = evaluasi_model("DummyClassifier (Baseline)", pipe_dummy, X_test, y_test)

### 6.12 Confusion Matrix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_lr,
    display_labels=['Tepat Waktu', 'Terlambat'],
    cmap='Blues', ax=axes[0]
)
axes[0].set_title('LogisticRegression')

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_dummy,
    display_labels=['Tepat Waktu', 'Terlambat'],
    cmap='Oranges', ax=axes[1]
)
axes[1].set_title('DummyClassifier (Baseline)')

plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
print("Confusion matrix disimpan sebagai 'confusion_matrix.png'")
plt.show()

### 6.13 Ekspor Hasil ke CSV

In [ ]:
ringkasan_bulanan.to_csv('ringkasan_bulanan.csv', index=False)
print("Tabel ringkasan disimpan sebagai 'ringkasan_bulanan.csv'")

metrik = pd.DataFrame({
    'Model': ['LogisticRegression', 'DummyClassifier'],
    'Akurasi': [accuracy_score(y_test, y_pred_lr),
                accuracy_score(y_test, y_pred_dummy)],
    'Precision': [precision_score(y_test, y_pred_lr, zero_division=0),
                  precision_score(y_test, y_pred_dummy, zero_division=0)],
    'Recall': [recall_score(y_test, y_pred_lr, zero_division=0),
               recall_score(y_test, y_pred_dummy, zero_division=0)],
    'F1': [f1_score(y_test, y_pred_lr, zero_division=0),
           f1_score(y_test, y_pred_dummy, zero_division=0)]
}).round(4)

metrik.to_csv('metrik_evaluasi.csv', index=False)
print("Metrik evaluasi disimpan sebagai 'metrik_evaluasi.csv'")
print()
metrik

## 6. Pembahasan

### Menjawab Rumusan Masalah 1: Mengapa kenaikan penjualan tidak selalu diikuti kenaikan laba?

Dari analisis deskriptif, terlihat bahwa total penjualan yang tinggi pada bulan tertentu tidak selalu diikuti laba yang tinggi. Ini disebabkan oleh:
- **Variasi harga pokok** yang berkisar 60-85% dari nilai penjualan — transaksi dengan rasio HPP tinggi menghasilkan margin tipis.
- **Biaya operasional** yang tidak selalu proporsional dengan penjualan — ada transaksi kecil dengan biaya operasional relatif besar.
- Akibatnya, beberapa transaksi menghasilkan **laba negatif**, yang menurunkan total laba meskipun penjualan tinggi.

Ini sesuai dengan penjelasan pemateri: *"Satu angka yang terlihat positif belum cukup untuk menggambarkan keadaan secara menyeluruh."*

### Menjawab Rumusan Masalah 2: Pelanggan mana yang berisiko terlambat membayar?

Model LogisticRegression diharapkan menunjukkan performa lebih baik dibanding DummyClassifier (baseline). Namun:
- **DummyClassifier** selalu memprediksi kelas mayoritas (tepat waktu), sehingga memiliki recall 0 untuk kelas terlambat.
- **LogisticRegression** mampu menangkap sebagian pelanggan berisiko, meskipun recall-nya mungkin tidak tinggi karena data sintetis dan fitur terbatas.

### Batasan Rekomendasi
- Model hanya menggunakan 3 fitur sederhana — dalam praktik nyata diperlukan fitur tambahan.
- Rekomendasi model bersifat **pendukung keputusan**, bukan pengganti penilaian manusia.
- Sesuai pesan pemateri: *"Menghasilkan prediksi belum sama dengan menentukan solusi."*

## 7. Keterbatasan dan Kesimpulan

### Keterbatasan
1. **Data sintetis** — Pola dalam data ini tidak mencerminkan kondisi bisnis nyata.
2. **Fitur terbatas** — Hanya 3 fitur digunakan untuk prediksi.
3. **Satu algoritma** — Hanya LogisticRegression yang diuji.
4. **Tanpa validasi silang** — Evaluasi hanya pada satu split data.
5. **Angka hasil** bergantung pada seed dan data sintetis — tidak bisa digeneralisasi.

### Kesimpulan
1. Analisis deskriptif memperlihatkan bahwa **kenaikan penjualan tidak menjamin kenaikan laba**, sesuai contoh pemateri.
2. Model LogisticRegression sederhana mampu memberikan prediksi awal tentang risiko keterlambatan, meskipun performanya terbatas.
3. **DummyClassifier** menunjukkan bahwa model mayoritas tidak informatif — penting membandingkan dengan baseline.
4. Alur CRISP-DM membantu menjaga analisis tetap terstruktur.
5. Seperti ditekankan pemateri, kemampuan utama analis bukan sekadar menguasai perangkat lunak, tetapi **memahami masalah, menjaga kualitas bukti, dan menjelaskan keterbatasan hasil**.